# Pseudo-3D por paralaxe (estilo fotos 3D do Facebook)

Este notebook mostra como obter um **movimento leve com sensação de profundidade** a partir de uma única imagem (`quadrado.png`), usando apenas **transformação geométrica** (remapeamento de pixels), sem filtros de cor.

**Ideia central:** nas fotos 3D do Facebook, existe um **mapa de profundidade** associado à imagem RGB. Ao inclinar o aparelho, cada camada se desloca horizontalmente em uma quantidade proporcional à profundidade — **paralaxe**. Aqui reproduzimos esse efeito com um mapa de profundidade **modelado geometricamente** (não estimamos profundidade com rede neural), de modo que **cada pixel da saída copia cores da entrada** por amostragem; a única alteração visível é o **deslocamento relativo** entre regiões próximas e distantes.

## 1. Conceitos

- **Mapa de profundidade** \(d(x,y)\): valor maior ≈ mais perto do observador (ou o inverso, desde que seja consistente).
- **Paralaxe horizontal:** para um ângulo de visão (ou “inclinação”) sintético \(\theta\), o deslocamento em colunas é proporcional a \((d - \bar d)\) vezes \(\sin(\theta)\): regiões com profundidade diferente da média **andam em velocidades diferentes**, criando volume.
- **Preservação de cor:** o pixel de saída \((y,x)\) lê o RGB da entrada em \((y, x + \Delta x)\) com \(\Delta x\) pequeno — **interpolação bilinear** entre vizinhos, sem mudar matiz ou contraste globalmente (apenas recombina pixels já existentes).

Limitação honesta: sem um mapa de profundidade real (LiDAR / dual-cam / estimativa monocular), o modelo de \(d\) é **hipotético**; ainda assim ilustra o algoritmo usado em visualizações tipo Facebook 3D.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from PIL import Image, ImageDraw
from scipy.ndimage import map_coordinates

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["image.interpolation"] = "nearest"

## 2. Entrada: `quadrado.png`

O arquivo deve estar na **mesma pasta** deste notebook (ou ajuste `CAMINHO_IMAGEM`). Se não existir, geramos um exemplo com um quadrado colorido só para você executar o fluxo completo.

In [ ]:
BASE = Path.cwd()
CAMINHO_IMAGEM = BASE / "quadrado.png"

if not CAMINHO_IMAGEM.is_file():
    w, h = 512, 512
    img_demo = Image.new("RGB", (w, h), (40, 45, 55))
    dr = ImageDraw.Draw(img_demo)
    margin = 96
    dr.rectangle(
        [margin, margin, w - margin, h - margin],
        fill=(220, 90, 70),
        outline=(255, 255, 200),
        width=6,
    )
    img_demo.save(CAMINHO_IMAGEM)
    print(f"Criado arquivo de demonstração: {CAMINHO_IMAGEM}")

im_pil = Image.open(CAMINHO_IMAGEM).convert("RGB")
im = np.asarray(im_pil, dtype=np.float32) / 255.0
H, W, _ = im.shape
print("Tamanho:", W, "x", H)

plt.imshow(im)
plt.title("Entrada: quadrado.png")
plt.axis("off")
plt.show()

## 3. Mapa de profundidade (modelo geométrico)

Usamos distância normalizada ao centro: o **centro** da imagem é tratado como mais **próximo** e as **bordas** mais **distantes** (como um relevo suave em forma de “tigela” invertida). Isso é uma escolha didática; para outras cenas você trocaria apenas a função `profundidade_radial`.

In [ ]:
def profundidade_radial(altura: int, largura: int) -> np.ndarray:
    """Retorna d em [0, 1]: 1 no centro, 0 nos cantos (após clip)."""
    yy, xx = np.mgrid[0:altura, 0:largura].astype(np.float32)
    cy, cx = (altura - 1) / 2.0, (largura - 1) / 2.0
    ny = (yy - cy) / max(cy, 1e-6)
    nx = (xx - cx) / max(cx, 1e-6)
    r = np.sqrt(nx * nx + ny * ny)
    rmax = np.sqrt(2.0)
    d = 1.0 - np.clip(r / rmax, 0.0, 1.0)
    return d


d = profundidade_radial(H, W)
plt.imshow(d, cmap="gray")
plt.colorbar(label="d (próximo → claro)")
plt.title("Mapa de profundidade sintético")
plt.axis("off")
plt.show()

## 4. Transformação: warping por paralaxe

Para cada frame, definimos um deslocamento horizontal **por pixel**:

$$\Delta x(y,x) = A\,(d(y,x) - 0.5)\,\sin(\phi)$$

onde \(A\) controla a intensidade e \(\phi\) simula a inclinação da “câmera” no tempo. A saída amostra `im[y, x + Δx]` com interpolação bilinear (**ordem 1** no `map_coordinates`). Os três canais usam o **mesmo** campo de deslocamento, logo não há mistura artificial entre canais além do que a geometria já faz.

In [ ]:
def paralaxe_horizontal(
    rgb: np.ndarray,
    depth: np.ndarray,
    fase: float,
    amplitude_px: float,
) -> np.ndarray:
    """
    rgb: float32 [H,W,3] em [0,1]
    depth: float32 [H,W] em [0,1]
    """
    h, w, _ = rgb.shape
    rel = depth.astype(np.float32) - 0.5
    shift = amplitude_px * rel * float(np.sin(fase))

    row, col = np.mgrid[0:h, 0:w].astype(np.float32)
    col_src = col + shift
    row_src = row

    out = np.empty_like(rgb)
    for c in range(3):
        out[..., c] = map_coordinates(
            rgb[..., c],
            [row_src, col_src],
            order=1,
            mode="nearest",
            prefilter=False,
        )
    return np.clip(out, 0.0, 1.0)


AMP = min(W, H) * 0.04  # ~2% do lado menor; ajuste para mais/menos movimento
phi_test = 0.35
warped = paralaxe_horizontal(im, d, phi_test, AMP)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(im)
ax[0].set_title("Original")
ax[1].imshow(warped)
ax[1].set_title(f"Com paralaxe (φ={phi_test:.2f})")
ax[2].imshow(np.abs(warped - im).max(axis=2), cmap="hot")
ax[2].set_title("Max |ΔRGB| por pixel")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

O terceiro painel mostra onde a geometria **reorganizou** pixels (diferença máxima entre canais). Onde \(d\) é quase constante, \(\Delta x \approx 0\) e a diferença tende a zero — ou seja, **sem mudança de cor além do que o remapeamento exige**.

## 5. Animação: movimento leve no tempo

Varrendo \(\phi\) no tempo obtemos o efeito de “inclinar” a cena de um lado para o outro.

In [ ]:
N_FRAMES = 48
fases = np.linspace(0, 2 * np.pi, N_FRAMES, endpoint=False)
frames = np.stack(
    [paralaxe_horizontal(im, d, ph, AMP) for ph in fases],
    axis=0,
)

fig, ax = plt.subplots()
ax.axis("off")
im_artist = ax.imshow(frames[0])


def atualizar(i):
    im_artist.set_array(frames[i])
    return (im_artist,)


anim = animation.FuncAnimation(
    fig,
    atualizar,
    frames=N_FRAMES,
    interval=45,
    blit=True,
)
plt.close(fig)
display(HTML(anim.to_jshtml()))

## 6. Exportar GIF (opcional)

Requer `pillow` já usado acima; o GIF quantiza cores — use a animação em HTML acima para fidelidade máxima. Mesmo assim o GIF pode ser útil para compartilhar.

In [ ]:
GIF_PATH = BASE / "quadrado_paralaxe.gif"
pil_frames = [
    Image.fromarray((np.clip(frames[i], 0, 1) * 255).astype(np.uint8))
    for i in range(N_FRAMES)
]
pil_frames[0].save(
    GIF_PATH,
    save_all=True,
    append_images=pil_frames[1:],
    duration=45,
    loop=0,
    optimize=False,
)
print("Salvo:", GIF_PATH.resolve())

## 7. Resumo

| Etapa | Papel |
|-------|--------|
| RGB de entrada | Textura sem alteração intencional de cor |
| Mapa \(d\) | Geometria 3D aproximada (aqui: radial ao centro) |
| Warp \(\Delta x \propto (d-0.5)\sin\phi\) | Paralaxe estilo fotos 3D |
| Bilinear | Amostragem suave; cores vêm só da imagem original |

Para aproximar melhor o Facebook em fotos reais, substitua `profundidade_radial` por um **mapa de profundidade** obtido do dispositivo ou por um modelo monocular — o bloco `paralaxe_horizontal` permanece o mesmo.